In [1]:
#import library
import pandas as pd
from pathlib import Path
import re

In [2]:
file_path = Path("messages.csv")

df = pd.read_csv(
    file_path,
    engine="python",
    on_bad_lines="warn"
)

In [ ]:
# display full text
pd.set_option("display.max_colwidth", None)  
pd.set_option("display.max_rows", None)      
pd.set_option("display.max_columns", None)  
pd.set_option("display.width", None) 

# I. Insight tập dữ liệu (Phần 1)

## 1.1 Insight từ những tin đăng mua

Dealer đồng hồ thường xuyên sử dụng thuật ngữ, ngôn ngữ ngầm định của giới đồng hồ

Anh Long cung cấp insight muốn mua thì dealer hay nhắn các thuật ngữ: SOLD ORDER = WANT TO BUY = LOOKING FOR = NEED TO QUOTE (NTQ) -> đây là yếu tố quyết định lọc

Sau khi lọc bằng các từ khóa trên 1 lần và đọc lại tập dữ liệu bị loại bỏ. Ta thấy ngoài các từ khóa trên họ còn dùng thêm Confirm, confirm orders (có s và không có s), need, ...

**Kết luận**: Ta có thể dùng keyword pattern để bắt các tin nhắn dạng: `Looking for RM011`

In [9]:
TRADE_KEYWORDS_PATTERN = re.compile(
    r"""
    \b(?:
        WTB|BUY|WANTED|ISO|LF|NEED|ORDER|LTB|CONFIRM|LOOKING|GOOD PRICE|
        SOLD\s+ORDER|
        NEED\s+TO\s+QUOTE|NTQ|
        WANT\s+TO\s+BUY|
        LOOKING\s+FOR|
        CONFIRM\s+ORDERS?
    )\b
    """,
    re.IGNORECASE | re.VERBOSE
)


In [22]:
df_clone1 = df.copy()
mask = df_clone1["message"].astype(str).str.contains(
    TRADE_KEYWORDS_PATTERN,
    na=False
)

df_clone1 = df_clone1[mask].copy()
# in ra 30 bản ghi looking for được bắt
# display full text
pd.set_option("display.max_colwidth", None)  
pd.set_option("display.max_rows", None)      
pd.set_option("display.max_columns", None)  
pd.set_option("display.width", None) 
df_clone1.sample(30)

,sender_name,message,msg_time
8145,Bill,LOOKING FOR//NEED TO BUY\n7128/1R - 7128/1G\nNEED NEW 2026\nDEAL IN HONGKONG//DUBAI⁠​‌⁠​⁠‍‌⁠​‍​‍‍‍‌⁠‍‌‌​‌‌‌‍‌,2026-06-16 09:17:49.000
95986,Dominic Hackett,"Morning all,\nNeed advice on repairing the end link on a 62523-14 ladies Jubilee bracelet. Need to replace or make the missing YG part. Apart from wear, it’s in good, serviceable condition. \nThanks in advance for any help.. 🙏🏻",NaN
25352,Dain 100%,Looking for sold order\r\n5267/200A WHITE\r\nNeed new in 2026‌‍‍⁠⁠‍‍​‌​‍‍⁠⁠‌‍​​⁠‌​‌‌‌‍‌,2026-06-17 10:35:53.000
43202,Dain 100%,SOLD ORDER\r\n5267/200A GREEN\r\nNEED NEW IN 2026‍‌‍‍​⁠‌‌​⁠⁠​‍‌‍​​‌​‍​‌‌‌‍‌,2026-06-18 09:53:37.000
74373,Bill,SOLD ORDER \n5167A\nNEED NEW 2026​‌​⁠‍​​​⁠​​​‌‌‌‌‍​​⁠​‌‌‌‍‌,2026-06-21 12:04:51.000
78704,Dain 100%,SOLD ORDER\r\n5822P\r\nNEED NEW IN 2025/2026‌‌‍​‌‍⁠‌​‌​⁠‌⁠⁠‍⁠‍​⁠​‌‌‌‍‌,2026-06-21 23:18:34.000
2589,Kato,Looking for 26240ST Black new 2026⁠‌‌‍‍‍‍‌‌‌‌‍‌​​​‍‌‌‌​‌‌‌‍‌,2026-06-16 02:50:32.000
51917,Kato,Looking for 5236P Blue - 5326P Salmon new 2026⁠​‌⁠​‌‍​‍⁠​​​​⁠​‍⁠​‍​‌‌‌‍‌,2026-06-18 21:52:10.000
21187,T&DJew,Looking for 5990.1r new or used 2024+ CFM All​‌‍⁠‍⁠‌⁠​‍‌⁠⁠‍‌‍⁠‍‍‌​‌‌‌‍‌,2026-06-17 04:37:03.000
63290,Kato,Looking for 5320G salmon new 2026‌⁠‍‍⁠‍​‍​‍⁠⁠​‍‌​⁠​‍‍​‌‌‌‍‌,2026-06-19 22:48:59.000


Cách dùng `pattern keyword` sẽ fail ở những case có keyword và đủ độ dài như ví dụ:

`ex`: Morning all,\nNeed advice on repairing the end link on a 62523-14 ladies Jubilee bracelet. Need to replace or make the missing YG part. Apart from wear, it’s in good, serviceable condition. \nThanks in advance for any help.. 🙏🏻

<i>Tin nhắn trên sửa dây rolex, chứ không phải bán đồng hồ</i>

## 1.2. Insight từ những tin đăng bán

Các tin đăng bán đều có đặc điểm chung là có giá: `1.04m`, `134k`, có currency `usd`, `usdt`, nếu không có giá thì chỉ có một case duy nhất là `good price` là một insight mạnh dùng để lọc data rác. Ta có thể viết ra `pattern price`

Tương tự như `pattern keyword`, `pattern price` sẽ cho pass những data rác giá và đủ ký tự: `we have 14k user in this group`

# 1.3 Kết luận
Có 2 phương án lọc data rác khả thi:

- Phương án 1: Train một model Machine learning phân biệt tin nhắn trade đồng hồ và non-trade đồng hồ. Cách này cũng dễ nhưng số lượng nhãn trade/non-trade chênh lệch cực đoan (9/1)  -> model dễ overfit và chi phí không rẻ
- Phương án 2: Dùng rule-base, xây các pattern như keyword, price -> khó xây một rule hoàn hảo cho mọi tin nhắn, khi rule quá dài thì ảnh hưởng đến hiệu năng lọc tin nhắn

# II. Insight tập dữ liệu (Phần 2)

Phần này sẽ liên quan nhiều đến thuật ngữ và domain đồng hồ hơn

## 2.1. Danh sách các thuật ngữ

Dealer nhắn: `12345 blue N6/26 130k usd`

Từ khóa N6/26 để chỉ năm sản xuất đồng hồ dựa theo danh sách dưới đây:

- N6/26: tháng 6 năm 2026
- N11/25: tháng 11 năm 2025.
- 24Y/Y24: năm 2024
- N1125: Tháng 11 năm 2025
- N4: Tháng 4 năm nay
- 2025y: 2025
...

Các thuật ngữ vô cùng đa dạng, bên dưới chưa phải là tất cả, có thể tạm phân vào các nhóm:

### Nhóm 1: Tình trạng đồng hồ:

- new sticker, new (ám chỉ đồng hồ mới)
- unworn (ám chỉ chưa đeo nhưng không có nghĩa là còn nguyên si)
- Like new (ám chỉ đã sử dụng nhưng còn như mới)
- used ( đã qua sử dụng)
- good condition (đã qua sử dụng nhưng điều kiện còn tốt)
- polished/Unpolished: chưa spa/ đã spa, đã đánh bóng
- mint/crisp: đã dùng nhưng tình trạng tốt, góc cạnh còn sắc bén)
- dented: cấn, móp
- scratched xtal/scratched crystal: kính bị xước
- watch and card: chỉ có đồng hồ và card
- damaged card: card bị hư hại
- NOS: new on stock, hàng tồn kho
  
#### Nhóm 2: Tình trạng giấy tờ & Phụ kiện đi kèm

- complete / full set: Sản phẩm đầy đủ paper/card, hộp
- naked - only watch: Chỉ có duy nhất đồng hồ, không có hộp, giấy -> giá giảm mạnh
- green card / hold card: Thuật ngữ đặc trưng của Rolex: Green Card: thẻ bảo hành của Rolex. Hold card: Thẻ bị đại lý giữ lại để chống đầu cơ
- service paper: Giấy chứng nhận bảo dưỡng/sửa chữa được cấp bởi trung tâm dịch vụ chính hãng, chứng minh đồng hồ chính hãng nếu bị mất paper
- no box: không có giấy tờ
- open Name, blank name: paper chưa điền tên (giá cao)
- blank date: paper chưa điền ngày
- NFC Card: card của rolex, quét chip là hiện thông tin đồng hồ
- Scratched
  
#### Nhóm 3: Tình trạng Thẻ định danh / Thẻ giá

- white tag: Tag bằng nhựa/giấy màu trắng của hãng, trên đó có in mã Ref, giá, seri đồng hồ
- no white tag: Bộ sản phẩm thiếu mất chiếc tag trắng này ->  giá giảm
- Green tag: Tag tròn màu xanh lá cây của Rolex đi kèm các mẫu đời mới, chứng nhận đồng hồ đạt chuẩn chính xác
- Both tags: có đầy đủ 2 loại tag

#### Nhóm 4: Nguồn gốc xuất xứ (Origin / Dealer Type)

- ADJ/Non ADJ: đồng hồ đã được điều chỉnh máy / chưa điều chỉnh máy

#### Nhóm 5: Tình trạng dây và Chi tiết đặc biệt

- full links: Dây đồng hồ còn đủ 100% số mắt dây tiêu chuẩn khi xuất xưởng (không bị cắt bớt)
- x links: số mắt dây để người mua tính xem có đeo vừa không 
- ghost: mặt số hoặc cái gì đó bị phai màu -> dealer săn đón

#### Nhóm 6: Thuật ngữ về giá

- Retail/MSRP: giá đăng bán bằng giá xuất xưởng
- ~ Retail: Xấp xỉ giá xuất xưởng
- Net / Net price: Giá về tay, người mua chịu phí ship
- wire ... price: Giá áp dụng khi thanh toán bằng hình thức chuyển khoản ngân hàng trực tiếp

## 2.2. Insight để đánh nhãn dữ liệu

Anh Long cung cấp insight: dealer chuyên nghiệp chỉ cần nhìn mã đồng hồ, giá + tình trạng đồng hồ là có thể ra quyết định có mua hay không

Sau khi thống kê các viết mã của dealer được phân thành 2 nhóm: 

- Nhóm 1: Mã đồng hồ được viết liền nhau `ví dụ`: Rolex: 13490g ( tôi fake thế :))) không vô được TimeDealer lấy mã thật)
- Nhóm 2: Mã đồng hồ được viết rời nhau `ví dụ` hãng RM: RM037 full di -> nó là một mã RM037 full diamond, hoặc hãng FPJ



<i>Rule bên mình giới hạn extract 16 hãng đồng hồ:</i>

- Mã đồng hồ hãng A. Lange & Söhne : 912.032

- Mã đồng hồ hãng Audemars Piguet : 14790BA.OO.0789BA.02

- Mã đồng hồ hãng Breguet: 9518ST/E2/984/D000

- Mã đồng hồ hãng Cartier : WSTA0053

- Mã đồng hồ Chopard : 68579-3004

- Mã đồng hồ F.P. Journe : FPJ Tourbillon Souverain Black Label

- Mã đồng hồ Hublot : 905.JX.0001.RT

- Mã đồng hồ Jacob & Co : Wh285G

- Mã đồng hồ Jaeger-LeCoultre : Q9088180

- Mã đồng hồ Omega : TI 145.0022 Alaska

- Mã đồng hồ Panerai : PAM36502

- Mã đồng hồ Patek Philippe: 2499J Series 2

- Mã đồng hồ Piaget: G0A41001

- Mã đồng hồ Rolex: 114060-0002

- Mã Vacheron Constantin: 89010/000P-9935

- Mã Richard Mille: RM001 Tourbillon

Trên thực tế, raw data có chứa cả đồng hồ các hãng như `Dior`, `Chanel`, nhưng extract data không có những hãng này. Nhãn được giới hạn 16 hãng này

# III. Hệ nhãn được chọn để đánh

Có 2 hệ nhãn thích hợp với kiểu dữ liệu là: `BIO` và `BIOES `

`BIOES` thì ưu điểm hơn vì pattern nhắn của khách có cấu trúc: <mã> <màu> <tình trạng đồng hồ> <giá>. Các từ này là các `single word`, nó không cần từ đứng phía sau để có đầy đủ ý nghĩa. Dùng hệ nhãn BIOES, model có thể kêt thúc sớm quá trình dự đoán nhãn

Dùng BIOES sẽ cần kiểm soát đánh nhãn gắt gao hơn

Hệ nhãn BIOES:

* **B (Begin):** Token bắt đầu của một thực thể có độ dài từ 2 token trở lên.
* **I (Inside):** Token nằm ở giữa của một thực thể (không phải đầu, không phải cuối).
* **O (Outside):** Token trung tính, không thuộc bất kỳ thực thể nào cần trích xuất (dấu câu, icon, từ nối...).
* **E (End):** Token kết thúc của một thực thể có độ dài từ 2 token trở lên.
* **S (Single):** Thực thể chỉ gồm duy nhất 1 token.


Hiện tại, Long đem tin nhắn dealer lên hỏi Gemini API với schema cầm được về:

```json
{
  "transaction": "forsale",
  "ref": "15210ST",
  "brand": "Audemars Piguet",
  "color": "Blue",
  "price": "143000",
  "currency": "USD",
  "year": "2023",
  "condition": "new",
  "note": "extra"
}

nên các đặc trưng bên dưới cần trích xuất


## 3.1 Các đặc trưng cần được trích xuất

Dựa trên đặc trưng dữ liệu, chúng ta thống nhất quản lý 6 loại thực thể sau:
1.  **REF:** Mã hiệu / Mã dòng đồng hồ (Mã ref).
2.  **COLOR:** Màu sắc của đồng hồ / mặt số / vòng bezel.
3.  **CONDITION:** Tình trạng đồng hồ (New, Used, Like new, Only watch, Fullset...).
4.  **PRICE:** Con số thể hiện giá tiền.
5.  **CURRENCY**:HKD, AED tệ (USD, VND, Triệu, Tr...).
6.  **NOTE**: -7% ( giảm giá) 
7.  **YEAR**: 2025y

Còn 2 thông tin sẽ khó đánh BIOES khó hơn một chút: đó là tin này thuộc loại giao dịch (B-TRANS) và brand (B-BRAND). Bởi vì dealer chỉ cần nhìn mã đồng hồ là đoán được brand rồi họ không hay nhắn brand. Cần dùng cách khác như lập từ điển mã đồng hồ - brand rồi dùng tìm kiếm fuzzy (điểm này có thể dùng masterfile của anh Đức). Các mã đồng hồ của từng hãng có một đặc trưng khác nhac nên fuzzy dùng ổn

Còn về đánh nhãn B-TRANS thì dealer muốn mua vào thường dùng các từ khóa `SOLD ORDER` `WANT TO BUY` `LOOKING FOR` `NEED TO QUOTE (NTQ)` nên trường này có thể dùng rule-based để check. Hơi khó xử lý ở phân loại 1 tin có cả tin bán và tin mua (chứa từ khóa). Tương lai 1 tin nhắn bao gồm tin mua + tin bán sẽ phải tách ra làm 2 phần

## 3.2 Các nhãn dữ liệu được chọn

LABELS:

`O`

`B-REF + I-REF + E-REF`: dùng cho mã viết cách
`S-REF`: dùng cho mã viết liền

`B-COLOR I-COLOR E-COLOR`: tôi cần verify lại xem màu dài nhất là 2 hay 3 word. nếu là 2 word thì bỏ nhãn I-COLOR đi
`S-COLOR`: dùng cho những màu red, blue

`B-PRICE E-PRICE`: 143 k (không có I-PRICE)
`S-PRICE`: 134k

`B-CONDITION I-CONDITION E-CONDITION`: phần này phân vân nhất, condition theo như code chỉ có những kiểu: used, new, like new quy về new thuộc condition

`S-YEAR`: đa số ký tự ngày, tháng được viết liền nhau

`S-CURRENCY`

`B-NOTE I-NOTE E-NOTE`: không phải những nhãn trên thì gắn hết thành Condition :)))

## 3.3 Chiến lược dùng AI gán nhãn

Vì mã đồng hồ cực kỳ quan trọng: những dòng Rolex, Hublot (mã viết liền nhau), dòng RM, FPJ (viết cách nhau) sẽ được gán nhãn theo từng đợt. Trong dòng Rolex cũng có người hay viết `1345 g`. (dù mã chuẩn viết liền nhau)

Lọc tin nhắn theo từng hãng đồng hồ, gán nhãn dữ liệu theo từng hãng đó (đem đi train + test từng loại đồng hồ). Ví dụ chỉ gán nhãn Rolex, chỉ train +  test bằng Rolex để đạt POS